In [1]:
import os
import time
import cv2
import numpy as np
import tensorrt as trt
from cuda.bindings import runtime as cudart
import os
import sys
import time
import cv2
import onnxruntime as ort
import platform
import subprocess
import torch
from PIL import Image
import numpy as np
from pathlib import Path
import torchvision.transforms as transforms
from configparser import ConfigParser
import re
import json
import threading
import pynvml
import tensorrt as trt
from cuda.bindings import runtime as cudart
import os
import time
import threading
import numpy as np
import tensorrt as trt
import pynvml


ALPHABET = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
PAD = "-"
IMG_WIDTH = 128
IMG_HEIGHT = 64

In [2]:
def get_current_vram(nvml_handle, pid):
    """Instantly snapshots the GPU memory used by the current process."""
    try:
        procs = pynvml.nvmlDeviceGetComputeRunningProcesses(nvml_handle)
        for p in procs:
            if p.pid == pid:
                return p.usedGpuMemory
    except Exception:
        pass
    return 0

In [3]:
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

In [4]:
project_dir=os.getcwd()
print(project_dir)

model_engine = "models/keras_model1.engine"
images = "test_images"

model_path_engine=os.path.join(os.getcwd(),model_engine)
image_folder=os.path.join(os.getcwd(),images)

print("model_path:- ",model_path_engine)
print("image_folder:-",image_folder)

/workspace
model_path:-  /workspace/models/keras_model1.engine
image_folder:- /workspace/test_images


In [5]:
def check_cuda(ret):
    err = ret[0] if isinstance(ret, tuple) else ret

    if err != cudart.cudaError_t.cudaSuccess:
        _, err_msg = cudart.cudaGetErrorString(err)
        raise RuntimeError(f"CUDA Error: {err_msg.decode()}")

In [6]:
def load_engine(engine_path):
    with open(engine_path, "rb") as f:
        runtime = trt.Runtime(TRT_LOGGER)
        engine = runtime.deserialize_cuda_engine(f.read())

    if engine is None:
        raise RuntimeError(f"Failed to deserialize TensorRT engine: {engine_path}")

    context = engine.create_execution_context()
    if context is None:
        raise RuntimeError("Failed to create TensorRT execution context.")

    err, stream = cudart.cudaStreamCreate()
    check_cuda(err)

    return engine, context, stream

In [7]:
def preprocess(folder_path):
    start_pre = time.perf_counter()
    image_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"))
    ])

    images = []
    for file in image_files:
        path = os.path.join(folder_path, file)
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_LINEAR)
        images.append(image)
    
    end_pre = time.perf_counter()
    preprocess_time_ms = (end_pre - start_pre) * 1000

    batch = np.stack(images).astype(np.float32)
    return batch, image_files,preprocess_time_ms


In [11]:


def inference_engine(engine, context, stream, batch_numpy, warmup_iters=10):
    has_nvml = False
    my_pid = os.getpid()
    nvml_handle = None
    
    try:
        pynvml.nvmlInit()
        nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        has_nvml = True
    except Exception:
        pass


    base_vram_bytes = get_current_vram(nvml_handle, my_pid) if has_nvml else 0
    base_vram_mb = base_vram_bytes / (1024 * 1024)
    print(f"Model VRAM Allocation           : {base_vram_mb:.2f} MB")

    # Setup shapes and data layouts
    input_name = engine.get_tensor_name(0)
    output_name = engine.get_tensor_name(1)

    batch_numpy = np.ascontiguousarray(batch_numpy, dtype=np.float32)
    context.set_input_shape(input_name, batch_numpy.shape)

    output_shape = tuple(context.get_tensor_shape(output_name))
    host_output = np.empty(output_shape, dtype=np.float32)

    # Allocate GPU Memory
    err, d_input = cudart.cudaMalloc(batch_numpy.nbytes)
    check_cuda(err)
    err, d_output = cudart.cudaMalloc(host_output.nbytes)
    check_cuda(err)

    # Set up tensor bindings before any execution
    context.set_tensor_address(input_name, int(d_input))
    context.set_tensor_address(output_name, int(d_output))

    # Active Background VRAM Tracking for Runtime Spikes
    peak_vram_bytes = base_vram_bytes
    stop_tracking = False

    def track_peak_memory():
        nonlocal peak_vram_bytes
        while not stop_tracking:
            current_mem = get_current_vram(nvml_handle, my_pid)
            if current_mem > peak_vram_bytes:
                peak_vram_bytes = current_mem
            time.sleep(0.001)  # Aggressive 1ms sampling window for inference spikes

    if has_nvml:
        mem_thread = threading.Thread(target=track_peak_memory, daemon=True)
        mem_thread.start()
        
    # Copy Input data to Device
    check_cuda(
        cudart.cudaMemcpyAsync(
            d_input,
            batch_numpy.ctypes.data,
            batch_numpy.nbytes,
            cudart.cudaMemcpyKind.cudaMemcpyHostToDevice,
            stream,
        )
    )
    
    # 2. Cold Inference Run
    start_cold = time.perf_counter()
    context.execute_async_v3(stream)
    check_cuda(cudart.cudaStreamSynchronize(stream))
    inference_cold_ms = (time.perf_counter() - start_cold) * 1000
    
    # 3. Warmup Iterations
    for _ in range(warmup_iters):
        context.execute_async_v3(stream)
    check_cuda(cudart.cudaStreamSynchronize(stream))
    
    # 4. Warm Inference Run (Timed with DtoH Copy)
    start_warm = time.perf_counter()
    context.execute_async_v3(stream)
    check_cuda(
        cudart.cudaMemcpyAsync(
            host_output.ctypes.data, 
            d_output, 
            host_output.nbytes, 
            cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost, 
            stream
        )
    )
    check_cuda(cudart.cudaStreamSynchronize(stream))
    inference_warm_ms = (time.perf_counter() - start_warm) * 1000

    print(f"Cold Inference (Without Warmup) : {inference_cold_ms:.2f} ms")
    print(f"Warm Inference (With Warmup)    : {inference_warm_ms:.2f} ms")

    # Clean up tracking thread
    stop_tracking = True
    if has_nvml:
        mem_thread.join()
        peak_gpu_usage_mb = peak_vram_bytes / (1024 * 1024)
        pynvml.nvmlShutdown()
    else:
        peak_gpu_usage_mb = 0.0

    # Cleanup GPU memory allocations
    cudart.cudaFree(d_input)
    cudart.cudaFree(d_output)

    return host_output, inference_cold_ms, inference_warm_ms, peak_gpu_usage_mb


In [12]:
def postprocess(outputs, filenames):
    start_time = time.perf_counter()
    
    results = []

    for file, pred in zip(filenames, outputs):
        indices = np.argmax(pred, axis=-1)
        scores = np.max(pred, axis=-1)

        plate = []
        confidence = []

        for idx, score in zip(indices, scores):
            if idx < len(ALPHABET):
                ch = ALPHABET[idx]
                if ch == PAD:
                    break
                plate.append(ch)
                confidence.append(score)

        plate_str = "".join(plate)
        mean_conf = float(np.mean(confidence)) if confidence else 0.0

        results.append({
            "image": file,
            "plate": plate_str,
            "confidence": mean_conf,
        })

    return results,(time.perf_counter() - start_time) * 1000

In [13]:
def main(model_path_engine, image_folder):
    engine, context, stream = load_engine(model_path_engine)
    batch, filenames, preprocess_ms = preprocess(image_folder)
    
    raw_predictions, inf_cold_ms, inf_warm_ms, peak_gpu_mb = inference_engine(
        engine, context, stream, batch, warmup_iters=1
    )
    # raw_predictions, inf_cold_ms, inf_warm_ms, peak_gpu_mb = inference_engine(batch, model_path_engine, warmup_iters=1)
    results, postprocess_ms = postprocess(raw_predictions, filenames)

        
    batch_size = len(filenames)
    total_run_time_warm_ms = preprocess_ms + inf_warm_ms + postprocess_ms
    
    print("=" * 60)
    print("TENSORRT RUNTIME PROFILE RUN SUMMARY")
    print("=" * 60)
    print(f"Total Images Processed : {batch_size}")
    print(f"Preprocess Latency     : {preprocess_ms / batch_size:.2f} ms per image")
    print(f"Inference (Warm Run)   : {inf_warm_ms / batch_size:.2f} ms per image")
    print(f"Postprocess Latency    : {postprocess_ms / batch_size:.2f} ms per image")
    print(f"Throughput             : {(batch_size / (total_run_time_warm_ms / 1000.0)):.2f} Images/sec")
    print(f"Model VRAM Allocation  : {peak_gpu_mb:.2f} MB")

    print("=" * 60)
    print("Result")
    print("=" * 60)
    for i in range(len(results)):
        print(results[i])
        print()
         
    
if __name__ == "__main__":
    main(model_path_engine, image_folder)



Model VRAM Allocation           : 0.00 MB
Cold Inference (Without Warmup) : 35.11 ms
Warm Inference (With Warmup)    : 6.25 ms
TENSORRT RUNTIME PROFILE RUN SUMMARY
Total Images Processed : 16
Preprocess Latency     : 4.72 ms per image
Inference (Warm Run)   : 0.39 ms per image
Postprocess Latency    : 0.05 ms per image
Throughput             : 193.78 Images/sec
Model VRAM Allocation  : 0.00 MB
Result
{'image': '234.jpeg', 'plate': 'AJ38BN8563', 'confidence': 0.5961562395095825}

{'image': '81aIaTwEBXL._AC_UF1000,1000_QL80_.jpg', 'plate': 'KA41CR4547', 'confidence': 0.8799319267272949}

{'image': 'IMG_20240120_151956.jpg', 'plate': 'RJ11DT3249', 'confidence': 0.9397172927856445}

{'image': 'Screenshot from 2026-08-05 13-16-41.png', 'plate': 'GJ03ER0563', 'confidence': 0.9881463050842285}

{'image': 'Screenshot from 2026-08-05 13-23-22.png', 'plate': 'MH02B14322', 'confidence': 0.8969478607177734}

{'image': 'Screenshot from 2026-08-05 13-23-44.png', 'plate': 'TN06BE3335', 'confidence': 